In [11]:
import boto3
import io
import glob
import logging
import math
import os
import requests
import s3fs
import subprocess
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

pl.Config.set_tbl_cols(-1)

polars.config.Config

- This notebook is meant to be run after `./get_gene_list.ipynb`

<br>

- This notebook performs the following steps:
<br>
<br>
    1. annotate gene list with C Marshall subpanel
    2. reorder cols in accordance with cols in `f"{REPO_ROOT}/notes/2026-10-15_c_trost_target_data_format.xlsx"`
    3. drop extraneous cols in accordance with cols in `f"{REPO_ROOT}/notes/2026-10-15_c_trost_target_data_format.xlsx"`
    3. drop rows with missing AA sequences


In [22]:
# ============================================================
#       Args
# ============================================================

# File intake
FILE_GENE_LIST = f"{REPO_ROOT}/results/2026-10-07_for_c_marshal.tsv"
FILE_MARSHALL_AI = f"{REPO_ROOT}/data_sync/c_marshal_ai_panel_genes.xlsx"
FILE_TARGET_SCHEMA = f"{REPO_ROOT}/notes/2026-10-15_c_trost_target_data_format.xlsx"

# Dirs
OUT_DIR = f"{REPO_ROOT}/results"

In [12]:
# ============================================================
#       In -- gene list
# ============================================================

df_gene_list = pl.read_csv(
    FILE_GENE_LIST,
    separator="\t",
    has_header=True,
    infer_schema_length=None,
    null_values="NA"
)

df_gene_list

defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,pymol_rmsd,pymol_aligned_atoms,pymol_aligned_residues,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date,human_gene_in_IUIS,defense_AA_sequence,human_AA_sequence,blastp_pident,blastp_length,blastp_mismatch,blastp_gapopen,blastp_qstart,blastp_qend,blastp_sstart,blastp_send,blastp_evalue,blastp_bitscore
str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,bool,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str,bool,str,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""Q47ZY9""","""A0A2E5CU08""",167879,"""Q8IZH9""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,177,null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…","""cd00642;""","""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,"""NP_001019195.1;""","""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…",1,202,"""DOMAIN 2..176; /note=""GTP cycl…","""Dpd__FolE""",0.991,0.964,0.51,0.28,986,175,1,176,6,181,0.0,false,182,"""DOMAIN 3..180; /note=""GTP cycl…",false,0,null,95.53,97.12,8.8818e-16,2.5535e-15,2.5535e-15,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9962,"""model_v6""","""""",null,"""MTIENNYREILATIGEDINRGGLLDTPKRA…","""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…",56.0,175,77,0,7,181,2,176,7.3000e-82,227.0
"""A0A5B9E1W6""","""A0A7C2NS94""",582702,"""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""",397,"""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDE""","""Rhodanobacter glycinis""","""GCF_008000795_NZ_CP042807_Dnd_…","""GCF_008000795.1_NZ_CP042807_02…","""GCF_008000795.1_NZ_CP042807_02…",1,202,"""DOMAIN 2..361; /note=""Aminotra…","""Dnd__DndA""",0.9237,0.9496,1.3,0.54,1874,367,1,376,9,386,0.0,false,386,"""DOMAIN 7..369; /note=""Aminotra…",false,0,null,94.52,95.11,1.3656e-14,4.6629e-15,1.3656e-14,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9863,"""model_v6""","""""",false,"""MSDGRPIYLDHNATTPLDPTVFEAMRPYFL…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…",44.18,378,208,3,9,385,1,376,5.3700e-114,326.0
"""Q47ZY9""","""A0A2E5CU08""",167879,"""Q8IZH8""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,158,null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [

In [ ]:
# ============================================================
#       In -- C Marshall panel
# ============================================================

df_marshall_panel = pl.concat([
    pl.read_excel(
        FILE_MARSHALL_AI,
        sheet_name=sheet
    ).with_columns(
        pl.lit(sheet).alias("source_sheet")
    )
    for sheet in ["RFS", "HLH", "AI", "AG"]
])

df_marshall_panel

Subpanel,snpEff_gene,snpEff_refseq,source_sheet
str,str,str,str
"""RFS""","""MEFV""","""NM_000243.2""","""RFS"""
"""RFS""","""MVK""","""NM_000431.3""","""RFS"""
"""RFS""","""NLRP12""","""NM_144687.3""","""RFS"""
"""RFS""","""NLRP3""","""NM_004895.4""","""RFS"""
"""RFS""","""TNFRSF1A""","""NM_001065.3""","""RFS"""
…,…,…,…
"""AG""","""RNASEH2A""","""NM_006397.2""","""AG"""
"""AG""","""RNASEH2B""","""NM_024570.3""","""AG"""
"""AG""","""RNASEH2C""","""NM_032193.3""","""AG"""


In [23]:
# ============================================================
#       In -- target schema
# ============================================================

target_schema_cols = pl.read_excel(FILE_TARGET_SCHEMA, has_header=True, read_options={"n_rows": 0}).columns
target_schema_cols


['composite_score',
 'defense_uniprot_ac',
 'human_uniprot_ac',
 'human_gene_name_primary',
 'human_gene_names',
 'griid_gene',
 'iuis_iei',
 'human_gene_in_IUIS',
 'uniprot_ref_proteome_chr',
 'human_protein_names',
 'Gene Ontology (molecular function)',
 'Gene Ontology IDs',
 'RefSeq',
 'defense_type',
 'defense_subtype',
 'defense_duplicate_count',
 'defense_gene_name',
 'defense_species',
 'human_length_AAs',
 'human_aligned_start',
 'human_aligned_end',
 'bacterial_length',
 'bact_aligned_start',
 'bact_aligned_end',
 'pymol_aligned_residues',
 'pymol_rmsd',
 'plddt_human_mean',
 'human_AA_sequence',
 'plddt_bacteria_mean',
 'defense_AA_sequence',
 'blastp_pident',
 'blastp_length',
 'blastp_mismatch',
 'blastp_gapopen',
 'blastp_qstart',
 'blastp_qend',
 'blastp_sstart',
 'blastp_send',
 'blastp_evalue',
 'blastp_bitscore']

In [ ]:
# ============================================================
#       Annotate Marshall subpanel status
# ============================================================

subpanel_name_mappings = {
    "RFS": "Recurrent Fever Syndrome",
    "HLH": "Hemophagocytic Lymphohistiocytosis",
    "AI": "Autoinflammatory Diseases",
    "AG": "Aicardi-Goutieres Syndrome"
}

df_gene_list_ann = df_gene_list.join(
    df_marshall_panel.select(
        pl.col("snpEff_gene").alias("human_gene_name_primary"),
        pl.col("Subpanel").alias("marshall_immune_subpanel")
    ),
    on="human_gene_name_primary",
    how="left"
)

n_panel_hits = df_gene_list_ann["marshall_immune_subpanel"].len() - df_gene_list_ann["marshall_immune_subpanel"].null_count()
logger.info(f"Number of Marshall panel hits: {n_panel_hits}")

df_gene_list_ann = df_gene_list_ann.with_columns(
    pl.col("marshall_immune_subpanel").str.replace_many(
        list(subpanel_name_mappings.keys()),
        list(subpanel_name_mappings.values())
    )
)

df_gene_list_ann.filter(
    pl.col("marshall_immune_subpanel").is_not_null()
)

INFO:__main__:Number of Marshall panel hits: 9


defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,pymol_rmsd,pymol_aligned_atoms,pymol_aligned_residues,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date,human_gene_in_IUIS,defense_AA_sequence,human_AA_sequence,blastp_pident,blastp_length,blastp_mismatch,blastp_gapopen,blastp_qstart,blastp_qend,blastp_sstart,blastp_send,blastp_evalue,blastp_bitscore,marshall_immune_subpanel
str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,bool,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str,bool,str,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,str
"""A0A3E5G8Z5""","""A0A7U2YKB2""",674529,"""Q9NSU2""","""reviewed""","""Three-prime repair exonuclease…","""TREX1""",314,"""TREX1""","""UP000005640: Chromosome 3""","""activation of immune response …","""cytoplasm [GO:0005737]; cytoso…","""3'-5' exonuclease activity [GO…","""cytoplasm [GO:0005737]; cytoso…","""GO:0000287; GO:0001568; GO:000…","""3.30.420.10:FF:000046;""",null,"""3.30.420.10;""","""IPR013520;IPR012337;IPR036397;…","""PTHR13058;PTHR13058:SF27;""",null,null,"""SM00479;""","""NP_009179.2 [Q9NSU2-2];NP_3385…","""Cas""","""CAS_Class2-Subtype-II-C""","""Bacteroides faecis""","""GCF_020091505_NZ_CP081916_CAS_…","""GCF_020091505.1_NZ_CP081916_03…","""GCF_020091505.1_NZ_CP081916_04…",1,202,null,"""DEDDh_I_II_III_IV_V_VI_1""",0.4818,0.8028,2.29,1.71,657,142,13,227,2,167,2.9730e-13,true,180,"""DOMAIN 3..169; /note=""Exonucle…",false,0,null,80.23,94.31,0.000001,2.0247e-12,0.000001,0.000034,"""afdb_v6""","""afdb_v6""","""ok""",true,0.8991,"""model_v6""","""""",true,"""MKNFAAIDFETANDKRTSVCSVGIVIVRNG…","""MGSQALPPGPMQTLIFFDMEATGLPFSQPK…",25.373,67,43,3,67,129,130,193,0.79,15.8,"""Aicardi-Goutieres Syndrome"""
"""A0A2X1XYZ4""","""A0A316L5U4""",54005,"""Q5TZT0""","""unreviewed""","""exodeoxyribonuclease III (EC 3…","""TREX1 hCG_1997849""",304,"""TREX1""",null,"""DNA repair [GO:0006281]""","""cytosol [GO:0005829]; nucleus …","""DNA binding [GO:0003677]; doub…","""cytosol [GO:0005829]; nucleus …","""GO:0003677; GO:0005634; GO:000…","""3.30.420.10:FF:000046;""",null,"""3.30.420.10;""","""IPR013520;IPR012337;IPR036397;…","""PTHR13058;PTHR13058:SF27;""",null,null,"""SM00479;""","""NP_009179.2;""","""Cas""","""CAS_Class1-Subtype-I-E""","""Peptoniphilus harei""","""GCF_016726485_NZ_CP068103_CAS_…","""GCF_016726485.1_NZ_CP068103_00…","""GCF_016726485.1_NZ_CP068103_00…",1,202,"""DOMAIN 3..207; /note=""Exonucle…","""DEDDh_I_II_III_IV_V_VI_1""",0.5068,0.83,2.34,1.42,647,152,2,223,11,177,6.2580e-12,false,178,"""DOMAIN 12..177; /note=""Exonucl…",false,0,null,78.7,92.78,4.1532e-7,6.5803e-13,4.1532e-7,0.000019,"""afdb_v6""","""afdb_v6""","""ok""",false,0.899,"""model_v6""","""""",true,"""MKKSIDKKEIKKYAVLDLETDGLNPFNDNI…","""MQTLIFFDMEATGLPFSQPKVTELCLLAVH…",null,null,null,null,null,null,null,null,null,null,"""Aicardi-Goutieres Syndrome"""
"""A0A2X1XYZ4""","""A0A316L5U4""",54005,"""Q6IAN5""","""unreviewed""","""exodeoxyribonuclease III (EC 3…","""TREX1""",304,"""TREX1""",

In [26]:
# ============================================================
#       Target schema: wrangle cols
# ============================================================

# In accordance with `target_schema_cols`, a list of str ...
#   1. drop extraneous cols not in `target_schema_cols`
#   2. reorder cols in accordance with `target_schema_cols`

df_gene_list_cl = df_gene_list_ann.select(
    [col for col in target_schema_cols if col in df_gene_list_ann.columns]
)

df_gene_list_cl.write_csv(
    f"{OUT_DIR}/2026-10-15_for_c_marshal.tsv",
    separator="\t",
    include_header=True,
    null_value="NA"
)


df_gene_list_cl

composite_score,defense_uniprot_ac,human_uniprot_ac,human_gene_name_primary,human_gene_names,griid_gene,human_gene_in_IUIS,uniprot_ref_proteome_chr,human_protein_names,Gene Ontology (molecular function),Gene Ontology IDs,RefSeq,defense_type,defense_subtype,defense_duplicate_count,defense_gene_name,defense_species,human_length_AAs,human_aligned_start,human_aligned_end,bacterial_length,bact_aligned_start,bact_aligned_end,pymol_aligned_residues,pymol_rmsd,plddt_human_mean,human_AA_sequence,plddt_bacteria_mean,defense_AA_sequence,blastp_pident,blastp_length,blastp_mismatch,blastp_gapopen,blastp_qstart,blastp_qend,blastp_sstart,blastp_send,blastp_evalue,blastp_bitscore
f64,str,str,str,str,bool,bool,str,str,str,str,str,str,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,f64,f64,str,f64,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64
0.9962,"""Q47ZY9""","""Q8IZH9""",null,null,false,null,null,"""GTP cyclohydrolase 1 (EC 3.5.4…","""GTP binding [GO:0005525]; GTP …","""GO:0003934; GO:0005525; GO:000…","""NP_001019195.1;""","""Dpd""","""Dpd""",1,"""Dpd__FolE""","""Colwellia psychrerythraea""",177,1,176,182,6,181,175,0.28,95.53,"""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…",97.12,"""MTIENNYREILATIGEDINRGGLLDTPKRA…",56.0,175,77,0,7,181,2,176,7.3000e-82,227.0
0.9863,"""A0A5B9E1W6""","""Q5QP19""","""NFS1""","""NFS1 hCG_38220""",false,false,null,"""Cysteine desulfurase (EC 2.8.1…","""cysteine desulfurase activity …","""GO:0005654; GO:0005829; GO:000…",null,"""Dnd""","""Dnd_ABCDE""",1,"""Dnd__DndA""","""Rhodanobacter glycinis""",397,1,376,386,9,386,367,0.54,94.52,"""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…",95.11,"""MSDGRPIYLDHNATTPLDPTVFEAMRPYFL…",44.18,378,208,3,9,385,1,376,5.3700e-114,326.0
0.9825,"""Q47ZY9""","""Q8IZH8""",null,null,false,null,null,"""GTP cyclohydrolase 1 (EC 3.5.4…","""GTP binding [GO:0005525]; GTP …","""GO:0003934; GO:0005525; GO:000…",null,"""Dpd""","""Dpd""",1,"""Dpd__FolE""","""Colwellia psychrerythraea""",158,1,156,182,6,164,146,0.38,89.02,"""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…",97.12,"""MTIENNYREILATIGEDINRGGLLDTPKRA…",57.463,134,57,0,7,140,2,135,1.2400e-63,180.0
0.9766,"""Q47ZY9""","""P30793""","""GCH1""","""GCH1 DYT5 GCH""",false,false,"""UP000005640: Chromosome 14""","""GTP cyclohydrolase 1 (EC 3.5.4…","""GTP binding [GO:0005525]; GTP …","""GO:0003924; GO:0003934; GO:000…","""NP_000152.1 [P30793-1];NP_0010…","""Dpd""","""Dpd""",1,"""Dpd__FolE""","""Colwellia psychrerythraea""",250,70,250,182,2,182,180,0.33,86.5,"""MEKGPVRAPAEKPRGARCSNGFPERDPPRP…",97.12,"""MTIENNYREILATIGEDINRGGLLDTPKRA…",56.0,175,77,0,7,181,75,249,1.1400e-80,227.0
0.9729,"""W5Y3L4""","""H0YG54""","""REXO2""","""REXO2""",false,false,"""UP000005640: Chromosome 11""","""Oligoribonuclease, mitochondri…","""3'-5'-RNA exonuclease activity…","""GO:0000175; GO:0003676; GO:000…",null,"""Cas""","""CAS_Cluster""",1,"""DEDDh_I_II_III_IV_V_VI_1""","""Corynebacterium vitaeruminis""",188,1,169,209,19,188,168,0.62,91.31,"""MTGLDIEKDQIIEMACLITDSDLNILAEGP…",94.35,"""MTEPVAPKNDRLVWIDLEMTGLDIHRHTIV…",47.305,167,87,1,19,185,1,166,3.5600e-55,161.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.8501,"""A0A250IWT1""","""B7ZLM5""","""ATG16L1""","""ATG16L1""",false,false,null,"""Autophagy-related protein 16-1…","""identical protein binding [GO:…","""GO:0000421; GO:0005765; GO:000…",null,"""Septu""","""Septu""",1,"""Septu__PtuA""","""Cystobacter fuscus""",561,265,560,759,424,716,293,1.88,84.98,"""QRQAFEEIILQYNKLLEKSDLHSVLAQKLQ…",53.22,"""MYVRSLTIQNLRSFESVELPLCHPGDGAPP…",24.042,287,199,7,422,697,264,542,8.6700e-20,79.7
0.85,"""L0D6C1""","""Q14498""","""RBM39""","""RBM39 HCC1 RNPC2""",false,false,"""UP000005640: Chromosome 20""","""RNA-binding protein 39 (CAPER …","""RNA binding [GO:0003723]; RS d…","""GO:0003723; GO:0005654; GO:000…","""NP_001229528.1 [Q14498-3];NP_0…","""Cas""","""CAS_Class1-Subtype-I-B""",1,"""cas6_I_II_III_IV_V_VI_21""","""Singulisphaera acidiphila""",530,150,392,209,23,202,35,2.04,66.79,"""MADDIDIEAMLEAPYKKDENKLSSANGHEE…",92.92,"""MPIIDLSFNIIGTTIPLDHGYSL